## 2026 CS3570 Introduction to Multimedia Technology

You need to design and apply different FIR filters into separate three audio signals from the given audio file (HW2_Mix.wav). Next, you are asked to reduce the sampling rate of filtered signals. Finally, since the output audio signals are too simple, you should apply one-fold echo and multiple-fold echo to produce more complex music.

Here's the step:
1. Transform the input signal into frequency domain and plot the spectrum (magnitude in spectrum should be normalized to [0, 1]).
2. Implement 3 different FIR filters to separate the three audio signals with Blackman window function (You have to pick the appropriate filter size, cut-off frequency, and window size).
3. Implement 1D convolution on the input signal with your filters (zero padding).
4. Reduce the sampling rates of the three separated songs to 2000Hz.
5. Apply one-fold echo and multiple-fold echo on the signal that pass through the **low-pass filter**. (Please use the audio files before reducing sampling rates)

### Reminder
* **The imported libraries are sufficient for this homework, you are not permitted to import other Python packages.**
* **Your code must display and output your results to enable us to verify its correctness.**
* **Please follow the instructions in the Jupyter Notebook and complete the parts marked as `"TODO."`**

In [ ]:
# pip install -r ../requirements.txt
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
from numpy.fft import fft
from tqdm import tqdm, trange

### 1. Plot the frequency spectrum

In [ ]:
def plot_spectrum(input_signal, fs, xbound, ybound, title):
    '''
    Input:
        input_signal: input signal (time-domain)
        fs: sampling frequency
        xbound: x-axis bound
        ybound: y-axis bound
        title: title of the plot
    '''
    # TODO #1 (1%): FFT & Plot the magnitude spectrum of the input signal
    # TODO #2 (1%): Comment your code!!
    # declare as float so FFT and normalization are stable
    input_signal = input_signal.astype(np.float64)

    # FFT of input signal
    spectrum = fft(input_signal)

    # for a real-valued signal, the spectrum is symmetric, so we only keep the positive-frequency half (https://math.stackexchange.com/questions/1271808/have-some-queries-about-fourier-transform)
    N = len(input_signal)
    half_N = N // 2

    # frequency axis (x axis) in Hz for the positive half
    freq = np.arange(half_N) * fs / N

    # magnitude spectrum (y axis)
    magnitude = np.abs(spectrum[:half_N])

    # normalize magnitude to [0, 1]
    if np.max(magnitude) != 0:
        magnitude = magnitude / np.max(magnitude)

    # combine frequency and normalized magnitude for plotting
    result = np.column_stack((freq, magnitude))

    print(result)

    plt.xlim(xbound[0], xbound[1])
    plt.ylim(ybound[0], ybound[1])

    plt.xlabel('frequency')
    plt.ylabel('magnitude')

    plt.plot(result[:,0], result[:,1])
    plt.title(title)
    plt.show()

### 2. Implement three different filters - Lowpass, Highpass, Bandpass



In [ ]:
def filter(fs, N, windowType, filterType, fc):
    '''
    Input:
        fs: sampling frequency
        N: filter length (number of points in a filter), also the window length
        windowType: window function type (Blackman)
        filterType: filter type (Low-pass, High-pass, Band-pass)
        fc: cutoff frequency
    Output:
        myFilter: low-pass, high-pass, or band-pass filter
    '''
    fc = fc/fs
    mid = np.floor(N/2).astype(int)
    samplepts = np.arange((-mid) , mid+1)
    samplepts[mid] = 1  ## prevent divide by 0
    myFilter = np.zeros(2 * mid + 1)

    if filterType == 'Low-pass':
        # TODO #3 (1%): Implement the Low-pass filter
        # TODO #4 (1%): Comment your code!!
        # h[n] = sin(2*pi*fc*n)/(pi*n), n != 0
        myFilter = np.sin(2 * np.pi * fc * samplepts) / (np.pi * samplepts)

        # center sample (n = 0) is the limit value 2*fc
        myFilter[mid] = 2 * fc

    elif filterType == 'High-pass':
        # TODO #5 (1%): Implement the High-pass filter
        # TODO #6 (1%): Comment your code!!
        # h[n] = -sin(2*pi*fc*n)/(pi*n), n != 0
        myFilter = -np.sin(2 * np.pi * fc * samplepts) / (np.pi * samplepts)
        myFilter[mid] = 1 - 2 * fc

    elif filterType == 'Band-pass':
        # TODO #7 (1%): Implement the Band-pass filter
        # TODO #8 (1%): Comment your code!!
        # for band-pass, fc contains two cutoff frequencies [f1, f2]
        f1 = fc[0]
        f2 = fc[1]

        # h[n] = sin(2*pi*f2*n)/(pi*n) - sin(2*pi*f1*n)/(pi*n), n != 0
        myFilter = (
            np.sin(2 * np.pi * f2 * samplepts) / (np.pi * samplepts)
            - np.sin(2 * np.pi * f1 * samplepts) / (np.pi * samplepts)
        )

        # center sample is 2*(f2 - f1)
        myFilter[mid] = 2 * (f2 - f1)


    if windowType == 'Blackman':
        # TODO #9 (1%): Implement Blackman window function
        # TODO #10 (1%): Comment your code!!

        # w[n] = 0.42 - 0.5*cos(2*pi*n/(N-1)) + 0.08*cos(4*pi*n/(N-1))
        # using a window makes the infinite ideal filter finite and reduces leakage
        n = np.arange(N)
        blackman_window = 0.42 - 0.5 * np.cos(2 * np.pi * n / (N - 1)) + 0.08 * np.cos(4 * np.pi * n / (N - 1))
        myFilter = myFilter * blackman_window

    return myFilter

In [ ]:
def plot_filter(filter, title):

    plt.figure()
    plt.plot(filter)
    plt.title(title)
    plt.xlabel('sample index n (discrete time)')
    plt.ylabel('amplitude')
    plt.show()

### 3. Implement convolution function

In [ ]:
def convolve(signal, filter, N):
    '''
    Input:
        signal: input signal (time-domain)
        filter: your filter
        N: filter length (number of points in a filter)
    Output:
        out: output signal (time-domain)
    '''
    len_signal = len(signal)
    out = np.zeros(len_signal)

    #TODO #11 (1%): Implement the 1D convolution operation
    #TODO #12 (1%): Comment your code!
    signal = signal.astype(np.float64) # filtered signal y
    filter = filter.astype(np.float64) # convolution mask h

    # zero-padding on both sides to compute convolution at the boundaries w/out indexing outside the signal
    pad = N // 2
    signal = np.concatenate((np.zeros(pad), signal, np.zeros(pad)))

    # for each output sample, take an N-point segment from the padded signal, multiply it with the filter, and sum them
    # FIR convolution formula
    for n in trange(len_signal):
      for k in range(N):
          out[n] += filter[k] * signal[n + pad - k]

    # convert back to float32 for saving audio
    out = out.astype(np.float32)
    return out

#TODO #13 and #14
* Summarize how you implemented the filter and convolutions to separate the mixed song. You may simply extract your explanation from the comments.
(1%)
* You should illustrate using anything you have written for Question 1-1. (1%)

### 4. Reduce sample rate

In [ ]:
def down_sampling(signal, fs, new_fs):
    '''
    Input:
        signal: input signal (time-domain)
        fs: sampling frequency
        new_fs: new sampling frequency
    Output:
        signal: down-sampled signal (time-domain)
    '''
    #TODO #15 (1%): Implement the down-sampling operation
    #TODO #15 (1%): Comment your code

    # take one sample every 'rate' steps
    # since fs isnt exactly divisible by new_fs, we use the closest integer rate
    rate = int(round(fs / new_fs))
    return signal[::rate]

### 5. Implement one-fold / multiple-fold echo

In [ ]:
def echo(signal_low, delay, alpha, fs):
    '''
    Input:
        signal_low: input signal (time-domain)
        delay: delay time
        alpha: attenuation factor
        fs: sampling frequency
    Output:
        output_echo_one: output signal with one-fold echo
        output_echo_multiple: output signal with multiple-fold echo
    '''
    D = int(delay * fs)
    input_len = len(signal_low)
    output_echo_one = np.zeros(input_len + D)

    #TODO #19 (1%): One-fold echo
    #TODO #20 (1%): Comment your code
    output_echo_one[:D] = signal_low[:D]

    for i in range(D, input_len + D): # we only start applying echo after delay D
        if i < input_len: # if original signal still exists, combine original + echo
            output_echo_one[i] = signal_low[i] + alpha * signal_low[i - D]
        else:
            # after original signal ends, only echo remains
            output_echo_one[i] = alpha * signal_low[i - D]

    D = int(delay * fs)
    input_len = len(signal_low)
    output_echo_multiple = np.zeros(input_len + D)

    #TODO 21 (1%): Multiple-fold echo
    #TODO 22 (1%): Comment your code
    output_echo_multiple[:D] = signal_low[:D]

    for i in range(D, input_len + D):
        if i < input_len:
            output_echo_multiple[i] = signal_low[i] + alpha * output_echo_multiple[i - D]
        else:
            # after original signal ends, recursive delayed echoes continue
            output_echo_multiple[i] = alpha * output_echo_multiple[i - D]

    output_echo_one = output_echo_one.astype(np.float32)
    output_echo_multiple = output_echo_multiple.astype(np.float32)

    return  output_echo_one, output_echo_multiple

### There should be 10 images displayed below the cell.



In [ ]:
from pathlib import Path
SRC_DIR = '.'
OUPUT_DIR = 'output'
Path(OUPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
import time
def main():
    audio = SRC_DIR + "/HW2_Mix.wav"
    fs, input_signal = wav.read(audio)

    # 1. Plot the magnitude spectrum of the input signal
    plot_spectrum(input_signal, fs, [0, 2000], [0, 2], "original_spectrum")

    # 2. implement filter
    #TODO #23 (1%): determine the cutoff frequency and window size
    #TODO #24 (1%): Explain how you determined the filter size and cut-off frequency.
    windowType = "Blackman"

    # I determined these by looking at the peaks in original_spectrum
    # also, after downsampling to 2000 Hz, the new Nyquist frequency is 1000 Hz, so each filtered song should ideally stay below 1000 Hz to avoid aliasing

    # low-frequency song
    fc_low = 400

    # higher-frequency song
    fc_high = 800

    # middle-frequency song
    fc_band = np.array([400, 800])

    # use odd filter size so the filter is symmetric and has a clear center
    # larger N -> sharper transition band, but more computation
    filterSize = 101

    lowpassFilter = filter(fs, N=filterSize, windowType=windowType, filterType='Low-pass', fc=fc_low)
    highpassFilter = filter(fs, N=filterSize, windowType=windowType, filterType='High-pass', fc=fc_high)
    bandpassFilter = filter(fs, N=filterSize, windowType=windowType, filterType='Band-pass', fc=fc_band)

    # plot filter shape and spectrum
    plot_filter(lowpassFilter, f"Low-pass_filter_{fc_low}")
    plot_filter(highpassFilter, f"High-pass_filter_{fc_high}")
    plot_filter(bandpassFilter , f"Band-pass_filter_{fc_band[0]}_{fc_band[1]}")
    plot_spectrum(lowpassFilter, fs, [0, 2000], [0, 2], f"Low-pass_spectrum_{fc_low}")
    plot_spectrum(highpassFilter, fs, [0, 2000], [0, 2], f"High-pass_spectrum_{fc_high}")
    plot_spectrum(bandpassFilter, fs, [0, 2000], [0, 2], f"Band-pass_spectrum_{fc_band[0]}_{fc_band[1]}")

    # 3. convolve the input signal with the filters
    # t = time.time()
    signal_low = convolve(input_signal, lowpassFilter, N=filterSize)
    signal_high = convolve(input_signal, highpassFilter, N=filterSize)
    signal_band = convolve(input_signal, bandpassFilter, N=filterSize)

    # save audio
    wav.write(OUPUT_DIR + f"/Low_pass_{fc_low}.wav", fs, signal_low)
    wav.write(OUPUT_DIR + f"/High_pass_{fc_high}.wav", fs, signal_high)
    wav.write(OUPUT_DIR + f"/Band_pass_{fc_band[0]}_{fc_band[1]}.wav", fs, signal_band)

    # plot output spectrum
    plot_spectrum(signal_low, fs, [0,2000], [0,2], "signal_low")
    plot_spectrum(signal_high, fs, [0,2000], [0,2], "signal_high")
    plot_spectrum(signal_band, fs, [0,2000], [0,2], "signal_band")

    # 4. Reduce sample rate
    new_fs = 2000
    signal_low_ds = down_sampling(signal_low, fs, new_fs)
    signal_high_ds = down_sampling(signal_high, fs, new_fs)
    signal_band_ds = down_sampling(signal_band, fs, new_fs)

    # You can also try setting 'new_fs' as 'fs' to see what happens.
    wav.write(OUPUT_DIR + f"/Low_pass_{fc_low}_2khz.wav", new_fs, signal_low_ds)
    wav.write(OUPUT_DIR + f"/High_pass_{fc_high}_2khz.wav", new_fs, signal_high_ds)
    wav.write(OUPUT_DIR + f"/Band_pass_{fc_band[0]}_{fc_band[1]}_2khz.wav", new_fs, signal_band_ds)

    # compare before and after downsampling
    plot_spectrum(signal_low_ds, new_fs, [0, 2000], [0, 1.1], "Low-pass after downsampling")
    plot_spectrum(signal_high_ds, new_fs, [0, 2000], [0, 1.1], "High-pass after downsampling")
    plot_spectrum(signal_band_ds, new_fs, [0, 2000], [0, 1.1], "Band-pass after downsampling")

    # 5. Echo
    echo_one , echo_multiple = echo(signal_low, 1, 0.5, fs)
    wav.write(OUPUT_DIR + '/echo_one.wav', fs, echo_one)
    wav.write(OUPUT_DIR + "/echo_multiple.wav", fs, echo_multiple)

if __name__ == "__main__":
    main()


#TODO #17 and #18

* Compare the difference between the signals before and after reducing the sampling rates (1%)
* You should illustrate using anything you have written for Question 1-1. (1%)


From the spectrum plots, the low-pass signal remains almost the same after downsampling, as its frequency components are mostly below 400 Hz. However, the high-pass signal shows significant changes after downsampling. This is because reducing the sampling rate lowers the Nyquist frequency to around 1000 Hz, causing higher-frequency components to be distorted or lost.

Listening to the audio also confirms that the high-pass signal is more affected than the low-pass signal, as can be heard from High_pass_800_2khz.wav that is more distorted

> High_pass_800.wav → original (less distorted), clear

> High_pass_800_2khz.wav → more distorted, weird




The band-pass signal shows some minor changes, but still preserves most of its structure since much of its frequency content is within the valid range.

This observation matches the Nyquist theorem explained in Question 1-1, where reducing the sampling rate lowers the maximum representable frequency. As shown in the spectrum plots, components above the new Nyquist frequency are either lost or distorted, especially in the high-pass signal.

#TODO #25

Compare the spectrum and shape of the filters (3%)

**All three filters are symmetric because they are FIR filters.**

The low-pass filter has a smooth bell-shaped curve, because it keeps slow changes in the signal (low frequencies).

The high-pass filter has a sharp center because it removes the smooth part and keeps rapid changes (high frequencies).

The band-pass filter looks more wavy because it only keeps a middle range of frequencies.

From the spectrum plots, the low-pass filter has high magnitude at low frequencies and decreases at higher frequencies, so it passes low-frequency components.

The high-pass filter has low magnitude at low frequencies and higher magnitude at high frequencies, so it passes high-frequency components.

The band-pass filter has higher magnitude only in the middle frequency range, so it passes frequencies between 400 Hz and 800 Hz while removing others.

Now, check your output directories, download all the wav files, play them, and include them in your HW2_Q1_(student ID)_(your name).zip.